# SemEval-2026 Task 2 & 3 (Track A: DimABSA)
# Subtask 2: DimASTE - Dimensional Aspect Sentiment Triplet Extraction
# Subtask 3: DimASQE - Dimensional Aspect Sentiment Quadruplets Extraction

-----


## Introduction:

You are welcome to participate in our SemEval Shared Task!

In this starter notebook, we guide you through the process of fine-tuning a pre-trained language model on sample training data to build a dimensional sentiment extraction model.  
This notebook is adapted from a HuggingFace-style implementation for similar tasks.

### Outline:
- Installation and importation of necessary libraries Setting up the project parameters. Running training and evaluation Before you start:

- It is strongly advised that you use a GPU to speed up training. To do this, go to the "Runtime" menu in Colab, select "Change runtime type" and then in the popup menu, choose "GPU" in the "Hardware accelerator" box.

### NB:
- This notebook aims to help you become familiar with fine-tuning language models for dimensional sentiment tasks.  
- You are encouraged to extend or modify it to obtain competitive performance.
- This notebook will take about 2 to 2.5 hours to run. It may shut down if your Colab GPU quota is insufficient.

### Languages and Domains:
#### Track A: Subtask 2 & 3
- eng_restaurant
- eng_laptop
- jpn_hotel
- rus_restaurant
- tat_restaurant
- ukr_restaurant
- zho_restaurant
- zho_laptop

### Model:

You can find the model here:
https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507-bnb-4bit

If you need alternative models (e.g., larger Qwen variants, or other multilingual LLMs), you may explore them on Hugging Face:
https://huggingface.co/unsloth/models

### Install unsloth package

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Load the competition data

In [2]:
import os
import json
from datasets import load_dataset

# Task Configuration
subtask = "subtask_2"
task = "task2"
lang = "eng"
domains = ["laptop", "restaurant"]

all_datasets = {}

for domain in domains:
    print(f"Loading data for {domain.upper()}...")

    # URLs
    train_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_alltasks.jsonl"
    dev_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl"
    test_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_test_{task}.jsonl"

    try:
        train_data = load_dataset("json", data_files=train_url, split="train")
        dev_data = load_dataset("json", data_files=dev_url, split="train")
        test_data = load_dataset("json", data_files=test_url, split="train")

        all_datasets[domain] = {
            "train": train_data,
            "validation": dev_data,
            "test": test_data
        }

        print(f"{domain.capitalize()} successfully loaded!")
        print(f"   - Train (Quadruplets): {len(train_data)} samples")
        print(f"   - Dev (Triplets): {len(dev_data)} samples")
        print(f"   - Test (Extraction): {len(test_data)} samples")

    except Exception as e:
        print(f"Error loading {domain} data: {e}")

Loading data for LAPTOP...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Laptop successfully loaded!
   - Train (Quadruplets): 4076 samples
   - Dev (Triplets): 200 samples
   - Test (Extraction): 1000 samples
Loading data for RESTAURANT...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Restaurant successfully loaded!
   - Train (Quadruplets): 2284 samples
   - Dev (Triplets): 200 samples
   - Test (Extraction): 1000 samples


In [3]:
gold_test_data = {}
unlabeled_test_data = {}

for domain in domains:
    try:
        test_ds = all_datasets[domain]["test"]

        # Gold Reference: Stores the original triplets for metric calculation
        gold_test_data[domain] = {
            sample["ID"]: sample.get("Triplet", []) for sample in test_ds}

        # Unlabeled Experiment Set: Strips all labels to simulate a raw test environment
        unlabeled_test_data[domain] = [
            {"ID": sample["ID"], "Text": sample["Text"]} for sample in test_ds]

        print(f" {domain.capitalize()} processed: {len(test_ds)} test samples separated.")

    except Exception as e:
        print(f" Error processing {domain}: {e}")

print("\nTest data separation complete for all domains.")

 Laptop processed: 1000 test samples separated.
 Restaurant processed: 1000 test samples separated.

Test data separation complete for all domains.


### Display the data info

In [4]:
for domain in domains:
    print(f"\n{'='*60}")
    print(f" DOMAIN: {domain.upper()} DATA PREVIEW")
    print(f"{'='*60}")

    # Display a sample from the Gold Reference (with labels)
    sample_id = unlabeled_test_data[domain][0]["ID"]
    print(f"\n[GOLD REFERENCE] (ID: {sample_id})")
    print(json.dumps(gold_test_data[domain][sample_id], indent=2))

    # Display a sample from the Unlabeled Test Data (without labels)
    print(f"\n[UNLABELED TEST DATA] (ID: {sample_id})")
    print(json.dumps(unlabeled_test_data[domain][0], indent=2))

    print(f"\n{'-'*60}")
    print(f"Total Inference Samples: {len(unlabeled_test_data[domain])}")


 DOMAIN: LAPTOP DATA PREVIEW

[GOLD REFERENCE] (ID: lap26_aste_test_1)
[
  {
    "Aspect": "battery life",
    "Opinion": "fairly low",
    "VA": "3.75#6.00"
  },
  {
    "Aspect": "laptop",
    "Opinion": "big",
    "VA": "4.50#6.50"
  }
]

[UNLABELED TEST DATA] (ID: lap26_aste_test_1)
{
  "ID": "lap26_aste_test_1",
  "Text": "Granted , the battery life is fairly low , and it's big for a laptop , but this is a desktop replacement , so that was to be expected"
}

------------------------------------------------------------
Total Inference Samples: 1000

 DOMAIN: RESTAURANT DATA PREVIEW

[GOLD REFERENCE] (ID: rest26_aste_test_1)
[
  {
    "Aspect": "curried chicken",
    "Opinion": "excellent",
    "VA": "8.25#8.12"
  },
  {
    "Aspect": "stewed chicken",
    "Opinion": "excellent",
    "VA": "8.25#8.12"
  }
]

[UNLABELED TEST DATA] (ID: rest26_aste_test_1)
{
  "ID": "rest26_aste_test_1",
  "Text": "My fiance had stewed chicken and I had the curried chicken- both were excellent"
}

--

### Design prompt template

In [5]:
from datasets import concatenate_datasets

# 1. The Unified Prompt Template
instruction = '''Below is an instruction describing a task, paired with an input that provides additional context. Your goal is to generate an output that correctly completes the task.

### Instruction:
Given a textual instance [Text], extract all (A, O, VA) triplets, where:
- A is an Aspect term (the entity being discussed)
- O is an Opinion term (the feeling or sentiment expressed about A)
- VA is a Valence–Arousal score in the format (valence#arousal)

Valence: 1.00 (negative) to 9.00 (positive).
Arousal: 1.00 (calm) to 9.00 (excited).
Important: Aspect and Opinion must maintain the exact case and spelling found in the [Text].

### Example:
Input:
[Text] average to good thai food, but terrible delivery.

Output:
[Triplet] (thai food, average to good, 6.75#6.38), (delivery, terrible, 2.88#6.62)

### Question:
Now complete the following example:
Input:
'''

# Training/Validation Conversion Function
def convert_for_training(x):
    text = x["Text"]
    source = x.get("Quadruplet") if "Quadruplet" in x else x.get("Triplet", [])
    answer_list = [f"({t['Aspect']}, {t['Opinion']}, {t['VA']})" for t in source]
    answer = "[Triplet] " + ", ".join(answer_list)
    return {"text": f"<|user|>\n{instruction}[Text] {text}\n\nOutput:\n<|assistant|>\n{answer}<|end_of_text|>"}

def prepare_inference_prompt(sample):
    text = sample["Text"]
    prompt = f"<|user|>\n{instruction}[Text] {text}\n\nOutput:\n<|assistant|>\n"
    return {"ID": sample["ID"], "prompt": prompt, "Text": text}

# DATASET MAPPING
print("Mapping training and validation sets...")

laptop_train_mapped = all_datasets['laptop']['train'].map(convert_for_training, remove_columns=all_datasets['laptop']['train'].column_names)
restaurant_train_mapped = all_datasets['restaurant']['train'].map(convert_for_training, remove_columns=all_datasets['restaurant']['train'].column_names)

laptop_dev_mapped = all_datasets['laptop']['validation'].map(convert_for_training, remove_columns=all_datasets['laptop']['validation'].column_names)
restaurant_dev_mapped = all_datasets['restaurant']['validation'].map(convert_for_training, remove_columns=all_datasets['restaurant']['validation'].column_names)

train_dataset = concatenate_datasets([laptop_train_mapped, restaurant_train_mapped])
eval_dataset = concatenate_datasets([laptop_dev_mapped, restaurant_dev_mapped])

inference_test_prompts = {
    domain: [prepare_inference_prompt(s) for s in unlabeled_test_data[domain]]
    for domain in domains
}

print(f"Training set ready: {len(train_dataset)} samples.")
print(f"Validation set ready: {len(eval_dataset)} samples.")
print(f"Inference prompts ready for {sum(len(v) for v in inference_test_prompts.values())} test samples.")

Mapping training and validation sets...


Map:   0%|          | 0/4076 [00:00<?, ? examples/s]

Map:   0%|          | 0/2284 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Training set ready: 6360 samples.
Validation set ready: 400 samples.
Inference prompts ready for 2000 test samples.


### Load the LLM from Hugging Face and apply LoRA for fine-tuning


In [6]:
# Install Unsloth for Colab
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-lzqgjszm/unsloth_be52fa55b1e24e92b6013279f5003bc5
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-lzqgjszm/unsloth_be52fa55b1e24e92b6013279f5003bc5
  Resolved https://github.com/unslothai/unsloth.git to commit 01f2e289a7376a3a4d71a2e39bed025a72df0273
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 25.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 16.9 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers

In [7]:
import os
import gc
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import get_token

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

domains = ["laptop", "restaurant"]
model_path = "unsloth/llama-3.1-8b-instruct-bnb-4bit"

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()

#  THE TRAINING LOOP ---
for domain in domains:
    print(f"\n" + "="*50)
    print(f"STARTING FINE-TUNING: Llama-3.1-8B ({domain.upper()})")
    print(f"="*50)

    clear_vram()

    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = model_path,
            max_seq_length = 1024,
            load_in_4bit = True,
            token = get_token()
        )

        model = FastLanguageModel.get_peft_model(
            model,
            r = 16,
            target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                              "gate_proj", "up_proj", "down_proj",],
            lora_alpha = 16,
            lora_dropout = 0,
            bias = "none",
            use_gradient_checkpointing = "unsloth",
            random_state = 3407,
        )

        # Using your previously mapped datasets
        current_train = laptop_train_mapped if domain == "laptop" else restaurant_train_mapped
        current_eval = laptop_dev_mapped if domain == "laptop" else restaurant_dev_mapped

        trainer = SFTTrainer(
            model = model,
            tokenizer = tokenizer,
            train_dataset = current_train,
            eval_dataset = current_eval,
            dataset_text_field = "text",
            max_seq_length = 1024,
            args = TrainingArguments(
                per_device_train_batch_size = 1,
                gradient_accumulation_steps = 8,
                warmup_steps = 5,
                max_steps = 300,
                learning_rate = 2e-4,
                fp16 = True,
                fp16_full_eval = True,
                per_device_eval_batch_size = 1,
                eval_accumulation_steps = 4,
                logging_steps = 10,
                eval_strategy = "steps",
                eval_steps = 50,
                eval_on_start = True,
                optim = "adamw_8bit",
                output_dir = f"outputs_Llama_{domain}",
                report_to = "none"
            ),
        )

        trainer.train()

        save_path = f"final_adapter_Llama_{domain}"
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"Successfully finished fine-tuning for {domain}!")

    except Exception as e:
        print(f"Error during {domain} training: {e}")

    # Aggressive cleanup
    if 'trainer' in locals(): del trainer
    if 'model' in locals(): del model
    clear_vram()

print("\nAll domain fine-tuning sessions are complete!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

STARTING FINE-TUNING: Llama-3.1-8B (LAPTOP)
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

...

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2284 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,284 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss,Validation Loss
0,No log,2.207052
50,0.203600,0.235872
100,0.223800,0.234130
150,0.203800,0.235175
200,0.176000,0.236345
250,0.172400,0.228786
300,0.178100,0.228933


Successfully finished fine-tuning for restaurant!

All domain fine-tuning sessions are complete!


### Run Inference and Extract Structured Sentiment Outputs

In [8]:
import json
import re
from unsloth import FastLanguageModel
import torch
import gc

# Regex utility to safely parse the model's generated text
def extract_triplets(generated_text):
    pattern = r'\(([^,]+),\s*([^,]+),\s*([\d.]+#[\d.]+)\)'
    matches = re.findall(pattern, generated_text)

    triplets = []
    for aspect, opinion, va in matches:
        triplets.append({
            "Aspect": aspect.strip(),
            "Opinion": opinion.strip(),
            "VA": va.strip()
        })
    return triplets

llama_method_a_results = {}
llama_terms_for_roberta = {}

for domain in domains:
    print(f"\n{'-'*50}")
    print(f"RUNNING INFERENCE: Llama-3.1-8B ({domain.upper()})")
    print(f"{'-'*50}")

    adapter_path = f"final_adapter_Llama_{domain}"

    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = adapter_path,
            max_seq_length = 1024,
            load_in_4bit = True,
        )
        FastLanguageModel.for_inference(model)

        domain_full_results = []
        domain_term_results = []

        total_samples = len(inference_test_prompts[domain])
        print(f"Processing {total_samples} samples for {domain}...")

        for i, sample in enumerate(inference_test_prompts[domain]):
            if i % 100 == 0:
                print(f"[{domain}] Processed {i}/{total_samples} samples...")

            inputs = tokenizer([sample["prompt"]], return_tensors="pt").to("cuda")

            # Generate the completion
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                use_cache=True,
                temperature=0.1,
                top_p=0.9
            )

            output_text = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
            triplets = extract_triplets(output_text)

            domain_full_results.append({
                "ID": sample["ID"],
                "Text": sample["Text"],
                "Triplet": triplets
            })

            # Save Term-Only Triplet (For RoBERTa Input)
            domain_term_results.append({
                "ID": sample["ID"],
                "Text": sample["Text"],
                "Terms": [{"Aspect": t["Aspect"], "Opinion": t["Opinion"]} for t in triplets]
            })

        llama_method_a_results[domain] = domain_full_results
        llama_terms_for_roberta[domain] = domain_term_results

        # Save results to JSON
        with open(f"llama_full_predictions_{domain}.json", "w") as f:
            json.dump(domain_full_results, f, indent=2)

        with open(f"llama_terms_only_{domain}.jsonl", "w") as f:
            for item in domain_term_results:
                f.write(json.dumps(item) + "\n")

        print(f"Saved predictions and term datasets for {domain}!")

    except Exception as e:
        print(f"Error during {domain} inference: {e}")

    finally:
        # Aggressive memory cleanup
        if 'model' in locals(): del model
        if 'tokenizer' in locals(): del tokenizer
        gc.collect()
        torch.cuda.empty_cache()

print("\nAll inference tasks complete! Files are ready next Step.")


--------------------------------------------------
RUNNING INFERENCE: Llama-3.1-8B (LAPTOP)
--------------------------------------------------
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Processing 1000 samples for laptop...
[laptop] Processed 0/1000 samples...
[laptop] Processed 100/1000 samples...
[laptop] Processed 200/1000 samples...
[laptop] Processed 300/1000 samples...
[laptop] Processed 400/1000 samples...
[laptop] Processed 500/1000 samples...
[laptop] Processed 600/1000 samples...
[laptop] Processed 700/1000 samples...
[laptop] Processed 800/1000 samples...
[laptop] Processed 9

### RoBERTa Large Training

In [9]:
import os
import json
import torch
import gc

os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()

model_name = "roberta-large"

def prepare_regression_dataset(original_dataset):
    """
    Extracts the text, aspect, opinion, and VA scores from your dataset.
    We format the text specifically to help RoBERTa understand what to look at.
    """
    data_dict = {"text": [], "labels": []}

    for sample in original_dataset:
        source = sample.get("Quadruplet") if "Quadruplet" in sample else sample.get("Triplet", [])

        for item in source:
            # RoBERTa uses </s> to separate parts of a sentence
            context = f"{sample['Text']} </s> {item['Aspect']} </s> {item['Opinion']}"

            # Split the VA string into two separate numbers
            v_str, a_str = item['VA'].split('#')
            v, a = float(v_str), float(a_str)

            data_dict["text"].append(context)
            data_dict["labels"].append([v, a]) # We train it to predict both at the same time!

    return Dataset.from_dict(data_dict)

for domain in domains:
    print(f"\n{'='*50}")
    print(f" TRAINING ROBERTA-LARGE REGRESSOR: {domain.upper()} ")
    print(f"{'='*50}")

    clear_vram()

    print("Preparing training and validation data...")
    train_ds = prepare_regression_dataset(all_datasets[domain]["train"])
    eval_ds = prepare_regression_dataset(all_datasets[domain]["validation"])

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

    tokenized_train = train_ds.map(tokenize_function, batched=True)
    tokenized_eval = eval_ds.map(tokenize_function, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        problem_type="regression"
    )

    # We use a small learning rate because RoBERTa-large is very sensitive
    training_args = TrainingArguments(
        output_dir=f"./roberta_regression_{domain}",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        fp16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        tokenizer=tokenizer,
    )

    print("Starting training...")
    trainer.train()
    print(f" Training complete for {domain}!")




 TRAINING ROBERTA-LARGE REGRESSOR: LAPTOP 
Preparing training and validation data...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-448286244.py:86: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer._unsloth___init__`. Use `processing_class` instead.
  trainer = Trainer(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,659 | Num Epochs = 5 | Total steps = 2,290
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 355,361,794 of 355,361,794 (100.00% trained)


Starting training...


Epoch,Training Loss,Validation Loss
1,No log,0.676385
2,2.451000,0.822886
3,0.479500,0.893986
4,0.332100,0.824893
5,0.238300,0.802136


 Training complete for restaurant!


**Prediction using RoBERTa**

In [23]:
import os
import json
import torch
import gc
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.trainer_utils import get_last_checkpoint

domains = ["laptop", "restaurant"]

for domain in domains:
    print(f"\n{'='*60}")
    print(f"prediction for {domain.upper()}...")
    print(f"{'='*60}")

    checkpoint_dir = f"./roberta_regression_{domain}"
    latest_checkpoint = get_last_checkpoint(checkpoint_dir)

    if latest_checkpoint is None:
        print(f"Error: Could not find checkpoints in {checkpoint_dir}.")
        print(f"Skipping {domain} prediction. Please check your file paths.")
        continue

    print(f"Found trained model at: {latest_checkpoint}")

    tokenizer = AutoTokenizer.from_pretrained("roberta-large")
    model = AutoModelForSequenceClassification.from_pretrained(latest_checkpoint).to("cuda")
    model.eval()

    try:
        with open(f"llama_terms_only_{domain}.jsonl", "r") as f:
            llama_terms_data = [json.loads(line) for line in f]
    except FileNotFoundError:
        print(f"Error: Could not find llama_terms_only_{domain}.jsonl. Skipping.")
        continue

    hybrid_predictions = []
    total_samples = len(llama_terms_data)

    # Prediction loop with progress tracking
    for i, sample in enumerate(llama_terms_data):
        if i % 100 == 0:
            print(f"[{domain}] Processed {i}/{total_samples} samples...")

        refined_triplets = []

        for term in sample["Terms"]:
            context = f"{sample['Text']} </s> {term['Aspect']} </s> {term['Opinion']}"
            inputs = tokenizer(context, return_tensors="pt", truncation=True, padding=True).to("cuda")

            with torch.no_grad():
                outputs = model(**inputs)
                scores = outputs.logits[0].cpu().numpy()

                # Keep scores strictly between 1.00 and 9.00
                v_score = max(1.0, min(9.0, float(scores[0])))
                a_score = max(1.0, min(9.0, float(scores[1])))

            refined_triplets.append({
                "Aspect": term["Aspect"],
                "Opinion": term["Opinion"],
                "VA": f"{v_score:.2f}#{a_score:.2f}"
            })

        hybrid_predictions.append({
            "ID": sample["ID"],
            "Triplet": refined_triplets
        })

    #Save the final Hybrid Method predictions
    with open(f"hybrid_full_predictions_{domain}.json", "w") as f:
        json.dump(hybrid_predictions, f, indent=2)

    print(f"Saved Hybrid predictions for {domain}!")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

print("\nPrediction complete for all domains!")


prediction for LAPTOP...
Found trained model at: ./roberta_regression_laptop/checkpoint-3610
[laptop] Processed 0/1000 samples...
[laptop] Processed 100/1000 samples...
[laptop] Processed 200/1000 samples...
[laptop] Processed 300/1000 samples...
[laptop] Processed 400/1000 samples...
[laptop] Processed 500/1000 samples...
[laptop] Processed 600/1000 samples...
[laptop] Processed 700/1000 samples...
[laptop] Processed 800/1000 samples...
[laptop] Processed 900/1000 samples...
Saved Hybrid predictions for laptop!

prediction for RESTAURANT...
Found trained model at: ./roberta_regression_restaurant/checkpoint-2290
[restaurant] Processed 0/1000 samples...
[restaurant] Processed 100/1000 samples...
[restaurant] Processed 200/1000 samples...
[restaurant] Processed 300/1000 samples...
[restaurant] Processed 400/1000 samples...
[restaurant] Processed 500/1000 samples...
[restaurant] Processed 600/1000 samples...
[restaurant] Processed 700/1000 samples...
[restaurant] Processed 800/1000 sampl

**Metrics Calculation and Comparision**

In [24]:
import json
import math

def calculate_va_dist(pred_va, gold_va):
    """
    Calculates normalized Euclidean distance between predicted and gold VA scores.
    Formula: sqrt((Vp-Vg)^2 + (Ap-Ag)^2) / sqrt(128)
    """
    try:
        vp, ap = map(float, pred_va.split('#'))
        vg, ag = map(float, gold_va.split('#'))
    except:
        # If Llama hallucinated a weird string format, it gets maximum distance (penalty)
        return 1.0

    d_max = math.sqrt(128)
    euclidean = math.sqrt((vp - vg)**2 + (ap - ag)**2)

    # Ensure distance is clamped between [0, 1]
    return min(euclidean / d_max, 1.0)

def compute_metrics(gold_data, pred_data):
    """
    Computes cPrecision, cRecall, and cF1.
    """
    total_ctp = 0
    total_preds = 0
    total_gold = 0

    for idx, golds in gold_data.items():
        preds = pred_data.get(idx, [])

        total_preds += len(preds)
        total_gold += len(golds)

        # Track matched gold elements to prevent double-counting
        matched_golds = set()

        for p in preds:
            best_ctp = 0
            best_gold_idx = -1

            for g_i, g in enumerate(golds):
                if g_i in matched_golds:
                    continue

                # Categorical Match: Aspect + Opinion must be identical (ignoring case/spaces)
                p_asp = p.get('Aspect', '').strip().lower()
                p_opi = p.get('Opinion', '').strip().lower()
                g_asp = g.get('Aspect', '').strip().lower()
                g_opi = g.get('Opinion', '').strip().lower()

                if p_asp == g_asp and p_opi == g_opi:
                    dist = calculate_va_dist(p.get('VA', '0#0'), g.get('VA', '0#0'))
                    ctp = 1.0 - dist

                    # If multiple match, pick the one that gives the best cTP
                    if ctp > best_ctp:
                        best_ctp = ctp
                        best_gold_idx = g_i

            if best_gold_idx != -1:
                total_ctp += best_ctp
                matched_golds.add(best_gold_idx)

    c_precision = total_ctp / total_preds if total_preds > 0 else 0
    c_recall = total_ctp / total_gold if total_gold > 0 else 0
    c_f1 = (2 * c_precision * c_recall) / (c_precision + c_recall) if (c_precision + c_recall) > 0 else 0

    return {"cPrecision": c_precision, "cRecall": c_recall, "cF1": c_f1}



In [25]:
# RUN COMPARISON EXPERIMENT
domains = ["laptop", "restaurant"]

print(f"\n{'='*70}")
print(f"{'FINAL EXPERIMENT RESULTS: LLAMA vs HYBRID ':^70}")
print(f"{'='*70}")

for domain in domains:
    print(f"\nDOMAIN: {domain.upper()}")

    gold_lookup = gold_test_data[domain]

    # Llama-Only Predictions
    try:
        with open(f"llama_full_predictions_{domain}.json", "r") as f:
            llama_raw = json.load(f)
            llama_lookup = {item["ID"]: item["Triplet"] for item in llama_raw}
    except FileNotFoundError:
        print(f"  [Warning] Could not find Llama predictions for {domain}.")
        llama_lookup = {}

    # Hybrid: Llama + RoBERTa
    try:
        with open(f"hybrid_full_predictions_{domain}.json", "r") as f:
            hybrid_raw = json.load(f)
            hybrid_lookup = {item["ID"]: item["Triplet"] for item in hybrid_raw}
    except FileNotFoundError:
        print(f"  [Warning] Could not find Hybrid predictions for {domain}.")
        hybrid_lookup = {}

    # Compute Metrics
    metrics_llama = compute_metrics(gold_lookup, llama_lookup)
    metrics_hybrid = compute_metrics(gold_lookup, hybrid_lookup)

    # Display Dashboard
    print(f"{'-'*70}")
    print(f"{'Method':<20} | {'cPrecision':<12} | {'cRecall':<12} | {'cF1 Score':<12}")
    print(f"{'-'*70}")
    print(f"{'A: Llama-Only':<20} | {metrics_llama['cPrecision']:.4f}       | {metrics_llama['cRecall']:.4f}       | {metrics_llama['cF1']:.4f}")
    print(f"{'B: Hybrid (RoBERTa)':<20} | {metrics_hybrid['cPrecision']:.4f}       | {metrics_hybrid['cRecall']:.4f}       | {metrics_hybrid['cF1']:.4f}")
    print(f"{'-'*70}")

    # Quick Conclusion
    diff = metrics_hybrid['cF1'] - metrics_llama['cF1']
    if diff > 0:
        print(f" Conclusion: The Hybrid method WON by +{diff:.4f} cF1.")
        print(f"   RoBERTa successfully corrected Llama's numerical VA inaccuracies.")
    elif diff < 0:
        print(f"Conclusion: Llama-Only WON by +{abs(diff):.4f} cF1.")
        print(f"   RoBERTa struggled to improve upon Llama's baseline VA guesses.")
    else:
        print("Conclusion: It's a tie! Both methods performed exactly the same.")


              FINAL EXPERIMENT RESULTS: LLAMA vs HYBRID               

DOMAIN: LAPTOP
----------------------------------------------------------------------
Method               | cPrecision   | cRecall      | cF1 Score   
----------------------------------------------------------------------
A: Llama-Only        | 0.5044       | 0.4439       | 0.4722
B: Hybrid (RoBERTa)  | 0.5015       | 0.4412       | 0.4694
----------------------------------------------------------------------
Conclusion: Llama-Only WON by +0.0028 cF1.
   RoBERTa struggled to improve upon Llama's baseline VA guesses.

DOMAIN: RESTAURANT
----------------------------------------------------------------------
Method               | cPrecision   | cRecall      | cF1 Score   
----------------------------------------------------------------------
A: Llama-Only        | 0.6039       | 0.5821       | 0.5928
B: Hybrid (RoBERTa)  | 0.5946       | 0.5731       | 0.5836
--------------------------------------------------------

# Subtask 3

In [13]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [14]:
import os
import re
import json
from datasets import load_dataset

data_urls = {
    "laptop": {
        "train": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_3/eng/eng_laptop_train_alltasks.jsonl",
        "dev": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_3/eng/eng_laptop_dev_task3.jsonl",
        "test": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_3/eng/eng_laptop_test_task3.jsonl"
    },
    "restaurant": {
        "train": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_3/eng/eng_restaurant_train_alltasks.jsonl",
        "dev": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_3/eng/eng_restaurant_dev_task3.jsonl",
        "test": "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_3/eng/eng_restaurant_test_task3.jsonl"
    }
}

all_datasets_task3 = {}

for domain in ["laptop", "restaurant"]:
    print(f"Loading Subtask 3 data for: {domain.upper()}...")

    all_datasets_task3[domain] = {
        "train": load_dataset("json", data_files=data_urls[domain]["train"], split="train"),
        "dev": load_dataset("json", data_files=data_urls[domain]["dev"], split="train"),
        "test": load_dataset("json", data_files=data_urls[domain]["test"], split="train")
    }

    print(f"{domain.capitalize()} - Train: {len(all_datasets_task3[domain]['train'])}, "
          f"Dev: {len(all_datasets_task3[domain]['dev'])}, "
          f"Test: {len(all_datasets_task3[domain]['test'])}")

print("\nSample Data Format (Laptop Train):")
print(all_datasets_task3["laptop"]["train"][0])

Loading Subtask 3 data for: LAPTOP...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Laptop - Train: 4076, Dev: 200, Test: 1000
Loading Subtask 3 data for: RESTAURANT...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Restaurant - Train: 2284, Dev: 200, Test: 1000

Sample Data Format (Laptop Train):
{'ID': 'laptop_quad_dev_1', 'Text': 'this unit is ` ` pretty ` ` and stylish , so my high school daughter was attracted to it for that reason .', 'Quadruplet': [{'Aspect': 'unit', 'Category': 'LAPTOP#DESIGN_FEATURES', 'Opinion': 'pretty', 'VA': '7.12#7.12'}, {'Aspect': 'unit', 'Category': 'LAPTOP#DESIGN_FEATURES', 'Opinion': 'stylish', 'VA': '7.12#7.12'}]}


In [15]:
print(all_datasets_task3)
print(all_datasets_task3['laptop']['train'][0])

{'laptop': {'train': Dataset({
    features: ['ID', 'Text', 'Quadruplet'],
    num_rows: 4076
}), 'dev': Dataset({
    features: ['ID', 'Text', 'Quadruplet'],
    num_rows: 200
}), 'test': Dataset({
    features: ['ID', 'Text', 'Quadruplet'],
    num_rows: 1000
})}, 'restaurant': {'train': Dataset({
    features: ['ID', 'Text', 'Quadruplet'],
    num_rows: 2284
}), 'dev': Dataset({
    features: ['ID', 'Text', 'Quadruplet'],
    num_rows: 200
}), 'test': Dataset({
    features: ['ID', 'Text', 'Quadruplet'],
    num_rows: 1000
})}}
{'ID': 'laptop_quad_dev_1', 'Text': 'this unit is ` ` pretty ` ` and stylish , so my high school daughter was attracted to it for that reason .', 'Quadruplet': [{'Aspect': 'unit', 'Category': 'LAPTOP#DESIGN_FEATURES', 'Opinion': 'pretty', 'VA': '7.12#7.12'}, {'Aspect': 'unit', 'Category': 'LAPTOP#DESIGN_FEATURES', 'Opinion': 'stylish', 'VA': '7.12#7.12'}]}


In [16]:


# SCHEMA DEFINITIONS ---
rest_entity = 'RESTAURANT, FOOD, DRINKS, AMBIENCE, SERVICE, LOCATION'
rest_attribute = 'GENERAL, PRICES, QUALITY, STYLE_OPTIONS, MISCELLANEOUS'

laptop_entity = 'LAPTOP, DISPLAY, KEYBOARD, MOUSE, MOTHERBOARD, CPU, FANS_COOLING, PORTS, MEMORY, POWER_SUPPLY, OPTICAL_DRIVES, BATTERY, GRAPHICS, HARD_DISK, MULTIMEDIA_DEVICES, HARDWARE, SOFTWARE, OS, WARRANTY, SHIPPING, SUPPORT, COMPANY'
laptop_attribute = 'GENERAL, PRICE, QUALITY, DESIGN_FEATURES, OPERATION_PERFORMANCE, USABILITY, PORTABILITY, CONNECTIVITY, MISCELLANEOUS'

def get_instruction(domain):
    entities = laptop_entity if domain == "laptop" else rest_entity
    attributes = laptop_attribute if domain == "laptop" else rest_attribute

    return f'''You are an expert Linguist specializing in Aspect-Based Sentiment Analysis (ABSA). Your task is to extract highly accurate (A, C, O, VA) quadruplets from the given text.

### **Core Extraction Rules:**
1. **Aspect (A):** The specific feature or entity mentioned. Must match the input text casing exactly.
2. **Category (C):** Classify the aspect using the "ENTITY#ATTRIBUTE" schema below. Use ONLY these labels. Must be UPPERCASE.
3. **Opinion (O):** The specific word/phrase used to express the sentiment. Match the input casing exactly.
4. **Valence-Arousal (VA):** - **Valence:** 1.00 (Extremely Negative) to 9.00 (Extremely Positive). 5.00 is Neutral.
   - **Arousal:** 1.00 (Calm/Sleepy) to 9.00 (Excited/Angry). 5.00 is Moderate.
   - Format as "V.VV#A.AA" (always 2 decimal places).

### **Label Schema Constraints:**
- **Valid Entities:** {entities}
- **Valid Attributes:** {attributes}

### **Step-by-Step Reasoning:**
Step 1: Identify all sentiment-bearing phrases and the aspects they refer to.
Step 2: Map each aspect to the most relevant ENTITY and ATTRIBUTE from the schema.
Step 3: Determine the numerical Valence (positivity) and Arousal (intensity) of the opinion.
Step 4: Format as a list of (A, C, O, VA) quadruplets.

---
### **Examples:**

**Input:** [Text] The screen is incredibly bright and vibrant, but the price is a bit steep.
**Output:** [Quadruplet] (screen, DISPLAY#QUALITY, incredibly bright and vibrant, 8.50#7.20), (price, LAPTOP#PRICE, a bit steep, 3.20#5.50)

**Input:** [Text] The waiter was polite but the food arrived cold.
**Output:** [Quadruplet] (waiter, SERVICE#GENERAL, polite, 7.00#4.50), (food, FOOD#QUALITY, cold, 2.50#6.80)

---
### **Target Task:**
Input:
[Text] {{input_text}}

Output:
'''



In [17]:
def convert_to_quads(x, domain):
    text = x["Text"]
    source = x.get("Quadruplet", [])

    # Format (Aspect, Category, Opinion, VA)
    quad_strings = [f"({q['Aspect']}, {q['Category']}, {q['Opinion']}, {q['VA']})" for q in source]
    answer = "[Quadruplet] " + ", ".join(quad_strings)

    # Build Prompt
    prompt = f"{get_instruction(domain)}{text}\n\nOutput:"

    return {"text": f"<|user|>\n{prompt}\n<|assistant|>\n{answer}<|end_of_text|>"}


# Process datasets
processed_datasets = {}
for domain in ["laptop", "restaurant"]:
    print(f"Loading and Mapping {domain.upper()}...")
    train_raw = load_dataset("json", data_files=data_urls[domain]["train"], split="train")
    dev_raw = load_dataset("json", data_files=data_urls[domain]["dev"], split="train")

    processed_datasets[domain] = {
        "train": train_raw.map(lambda x: convert_to_quads(x, domain), remove_columns=train_raw.column_names),
        "dev": dev_raw.map(lambda x: convert_to_quads(x, domain), remove_columns=dev_raw.column_names)
    }

Loading and Mapping LAPTOP...


Map:   0%|          | 0/4076 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Loading and Mapping RESTAURANT...


Map:   0%|          | 0/2284 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [18]:
import os
import torch
import gc
import re
import json
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import get_token

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

model_path = "unsloth/llama-3.1-8b-instruct-bnb-4bit"

for domain in ["laptop", "restaurant"]:
    print(f"\nSTARTING TASK 3 TRAINING: {domain.upper()}")
    clear_vram()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_path,
        max_seq_length = 1024,
        load_in_4bit = True,
        token = get_token()
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        use_gradient_checkpointing = "unsloth",
    )

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = processed_datasets[domain]["train"],
        eval_dataset = processed_datasets[domain]["dev"],
        dataset_text_field = "text",
        max_seq_length = 1024,
        args = TrainingArguments(
            per_device_train_batch_size = 1,
            gradient_accumulation_steps = 8,
            max_steps = 500,
            learning_rate = 2e-4,
            fp16 = True,
            fp16_full_eval = True,
            logging_steps = 10,
            eval_strategy = "steps",
            eval_steps = 50,
            optim = "adamw_8bit",
            output_dir = f"outputs_task3_{domain}",
            report_to = "none"
        ),
    )

    trainer.train()

    # Task 3 Adapters
    model.save_pretrained(f"task3_adapter_{domain}")
    tokenizer.save_pretrained(f"task3_adapter_{domain}")

    # Clean up before next domain
    del model, trainer
    clear_vram()

print("\nSubtask 3 Fine-Tuning Complete for all domains!")


STARTING TASK 3 TRAINING: LAPTOP
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4076 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,076 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


KeyboardInterrupt: 

In [ ]:
import re
import json
import torch
from unsloth import FastLanguageModel

def extract_quadruplets(text):
    result = []
    pattern = r'\(([^,]+),\s*([^,]+),\s*([^,]+),\s*([\d.]+#[\d.]+)\)'
    matches = re.findall(pattern, text)

    for aspect, category, opinion, va in matches:
        if "NULL" in aspect.upper() or "NULL" in opinion.upper():
            continue

        result.append({
            "Aspect": aspect.strip(),
            "Category": category.strip().upper(),
            "Opinion": opinion.strip(),
            "VA": va.strip()
        })
    return result

#Sequential Prediction Loop
all_task3_results = {}

for domain in ["laptop", "restaurant"]:
    print(f"\n" + "="*50)
    print(f"RUNNING TASK 3 INFERENCE: {domain.upper()}")
    print(f"="*50)

    # Load the domain-specific adapter
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = f"task3_adapter_{domain}",
        max_seq_length = 1024,
        load_in_4bit = True,
        device_map = {"": 0}
    )
    FastLanguageModel.for_inference(model)
    domain_instruction = get_instruction(domain)

    test_data = all_datasets_task3[domain]["test"]
    predictions = []

    for i, sample in enumerate(test_data):
        prompt = f"{domain_instruction}{sample['Text']}\n\nOutput:"

        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(inputs, return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
            temperature=0.1,
            eos_token_id=tokenizer.eos_token_id
        )

        # Decode Assistant response only
        input_len = inputs["input_ids"].shape[-1]
        decoded = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

        # Structure the final JSON
        dump_data = {
            "ID": sample["ID"],
            "Quadruplet": extract_quadruplets(decoded)
        }

        predictions.append(dump_data)

        if i % 50 == 0:
            print(f"[{domain}] {i}/{len(test_data)} processed.")

    all_task3_results[domain] = predictions

    # Clear memory for next domain
    del model, tokenizer
    torch.cuda.empty_cache()

print("\nTask 3 Extraction Complete!")

In [ ]:
import json
import os
import zipfile
from google.colab import files

# 1. Submission Configuration
subtask = "subtask_3"
lang = "eng"
domains = ["laptop", "restaurant"]

os.makedirs(subtask, exist_ok=True)


for domain in domains:
    # Resolve official filename
    out_name = f"pred_{lang}_{domain}.jsonl"
    jsonl_path = os.path.join(subtask, out_name)

    print(f"Writing {domain} predictions to {jsonl_path}...")

    results_list = all_task3_results.get(domain, [])

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for item in results_list:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

zip_name = f"{subtask}.zip"
print(f"Creating {zip_name}...")

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files_in_dir in os.walk(subtask):
        for file in files_in_dir:
            full_path = os.path.join(root, file)
            zf_path = os.path.relpath(full_path, ".")
            zf.write(full_path, zf_path)

# Browser download
print("Downloading submission file...")
files.download(zip_name)

print("\nSubtask 3 submission is ready! Good luck on the leaderboard!")